<a href="https://colab.research.google.com/github/ritashreemukherjee123/Project-1-Applied-Search-Intelligence-Google-Search-Ranking-Discoverability.-/blob/main/w03_feature_leakage_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

`dim_content` and `fact_daily` are the main data from the warehouse that we are using for the task of refresh/content opportunity scoring.

Building feature vectors

In [1]:
%pip -q install duckdb huggingface_hub


In [2]:
import os, getpass

# Token order: env var -> Colab Secret -> prompt (last resort).
# Use a Colab Secret named HF_TOKEN (the key panel on the left) so the prompt never
# fires: if Colab reconnects while a getpass prompt is open, the kernel waits on it
# forever ('Resuming execution...') and you have to restart the runtime.
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Ritashree')

Ritashree··········


In [3]:
##Connecting DuckDB to the release which would autheticate every query written
import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':                f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':                f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':                 f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample':          f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':             f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

for name, src in TABLES.items():
    n = con.sql(f'SELECT COUNT(*) FROM {src}').fetchone()[0]
    print(f'{name:22} {n:>12,} rows')


dim_clients                     104 rows
dim_content                 519,606 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_daily               78,835,655 rows
fact_daily_sample        11,694,072 rows
fact_query_90d            2,414,248 rows


In [4]:
import pandas as pd
from pathlib import Path
import duckdb
import os

# Constants
LOOKBACK_DAYS = 90
BASELINE_DAYS = 30
FUTURE_DAYS = 30
DECLINE_THRESHOLD = 0.70
DECISION_CADENCE_DAYS = 14

CACHE_PATH = Path("work/outputs/features.parquet")
CACHE_PATH.parent.mkdir(parents=True, exist_ok=True)
FEATURES_LOADED_FROM_CACHE = CACHE_PATH.exists()

if FEATURES_LOADED_FROM_CACHE:
    feature_vector_encoded = pd.read_parquet(CACHE_PATH)
    # Re-creating other DFs for consistency with the original notebook's flow if loaded from cache
    labeled = feature_vector_encoded.copy() # Simplified: In a real scenario, you might only need the final DF.
    historical_features = feature_vector_encoded.copy()
    feature_vector = feature_vector_encoded.copy()
    print(f"Loaded {len(feature_vector_encoded):,} cached feature rows from {CACHE_PATH}.")
else:
    print("Generating features from warehouse.")

    # Re-creating minimal decision_dates_sampled as its defining cell was deleted.
    # In a full run, this would be derived from fact_daily with specific windowing logic.
    # For demonstration, we'll take distinct report_dates from fact_daily_sample.
    decision_dates = con.sql(f"SELECT DISTINCT report_date AS decision_date FROM {TABLES['fact_daily_sample']} ORDER BY report_date").df()
    decision_dates_sampled = (
        decision_dates.sort_values("decision_date")
        .iloc[::DECISION_CADENCE_DAYS]
        .reset_index(drop=True)
    )
    print(f"Using {len(decision_dates_sampled)} sampled decision dates.")

    # 1. Historical Features
    historical_features = con.sql(f"""
    WITH daily AS (
        SELECT
            report_date, client_hash_id, content_hash_id,
            gsc_impressions, gsc_clicks,
            NULLIF(gsc_avg_position, 0) AS gsc_avg_position_clean,
            CASE WHEN ga4_data_available IS TRUE THEN ga4_sessions ELSE 0 END AS ga4_sessions_clean
        FROM {TABLES['fact_daily_sample']}
    )
    SELECT
        report_date AS as_of_date,
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions)        OVER w90      AS impressions_90d,
        SUM(gsc_impressions)        OVER w30      AS impressions_last30,
        SUM(gsc_impressions)        OVER w_prev30 AS impressions_prev30,
        SUM(gsc_clicks)             OVER w30      AS clicks_last30,
        SUM(gsc_clicks)             OVER w_prev30 AS clicks_prev30,
        AVG(gsc_avg_position_clean) OVER w30      AS avg_position_last30,
        SUM(ga4_sessions_clean)     OVER w30      AS ga4_sessions_last30,
        COUNT(*)                    OVER w90      AS active_days_90d
    FROM daily
    WINDOW
        w90      AS (PARTITION BY client_hash_id, content_hash_id ORDER BY report_date
                      RANGE BETWEEN INTERVAL {LOOKBACK_DAYS - 1} DAYS PRECEDING AND CURRENT ROW),
        w30      AS (PARTITION BY client_hash_id, content_hash_id ORDER BY report_date
                      RANGE BETWEEN INTERVAL {BASELINE_DAYS - 1} DAYS PRECEDING AND CURRENT ROW),
        w_prev30 AS (PARTITION BY client_hash_id, content_hash_id ORDER BY report_date
                      RANGE BETWEEN INTERVAL {2 * BASELINE_DAYS - 1} DAYS PRECEDING AND INTERVAL {BASELINE_DAYS} DAYS PRECEDING)
    QUALIFY as_of_date IN (SELECT decision_date FROM decision_dates_sampled)
    """).df()

    historical_features["ctr_last30"] = historical_features["clicks_last30"] / historical_features["impressions_last30"].replace(0, pd.NA)
    historical_features["ctr_prev30"] = historical_features["clicks_prev30"] / historical_features["impressions_prev30"].replace(0, pd.NA)
    historical_features["impression_ratio_last_vs_prev30"] = historical_features["impressions_last30"] / historical_features["impressions_prev30"].replace(0, pd.NA)
    print(f"{len(historical_features):,} (as_of_date, client, content) rows of history built.")

    # 2. Future Window for Label
    future_window = con.sql(f"""
        SELECT
            h.as_of_date, h.client_hash_id, h.content_hash_id,
            SUM(f.gsc_clicks) AS clicks_future30
        FROM historical_features h
        JOIN {TABLES['fact_daily_sample']} f
          ON f.client_hash_id = h.client_hash_id
         AND f.content_hash_id = h.content_hash_id
         AND f.report_date > h.as_of_date
         AND f.report_date <= h.as_of_date + INTERVAL {FUTURE_DAYS} DAY
        GROUP BY 1, 2, 3
    """).df()

    # 3. Label Creation
    labeled = historical_features.merge(
        future_window, on=["as_of_date", "client_hash_id", "content_hash_id"], how="left"
    )
    has_future_obs = labeled["clicks_future30"].notna()
    has_baseline = labeled["clicks_last30"] > 0
    labeled["is_declining"] = pd.NA
    valid_label = has_future_obs & has_baseline
    labeled.loc[valid_label, "is_declining"] = (
        labeled.loc[valid_label, "clicks_future30"]
        < labeled.loc[valid_label, "clicks_last30"] * DECLINE_THRESHOLD
    ).astype(int)

    # 4. Join with dim_content and initial feature processing (from deleted cell 67ef3ffbad58)
    feature_vector = con.sql(f"""
        SELECT
            lf.*,
            d.keyword_char_count,
            d.keyword_token_count,
            d.url_char_count,
            COALESCE(d.content_type, 'UNKNOWN_CONTENT_TYPE')             AS content_type,
            d.search_volume,
            d.competition,
            COALESCE(d.competition_level, 'UNKNOWN_COMPETITION_LEVEL')   AS competition_level,
            d.cpc,
            COALESCE(d.main_intent, 'UNKNOWN_MAIN_INTENT')               AS main_intent,
            d.backlinks,
            d.category_count,
            d.content_created_date
        FROM labeled lf
        LEFT JOIN {TABLES['dim_content']} d
          ON lf.client_hash_id = d.client_hash_id
         AND lf.content_hash_id = d.content_hash_id
        WHERE d.content_created_date IS NULL
           OR d.content_created_date <= lf.as_of_date
    """).df()

    # Keep only rows with a defined label
    feature_vector = feature_vector[feature_vector["is_declining"].notna()].reset_index(drop=True)
    feature_vector["is_declining"] = feature_vector["is_declining"].astype(int)

    # 5. Missing Numeric Fills & Flags
    numeric_cols_to_fill = [
        "search_volume", "competition", "cpc", "backlinks", "category_count",
        "keyword_char_count", "keyword_token_count", "url_char_count",
        "avg_position_last30", "ctr_last30", "ctr_prev30", "impression_ratio_last_vs_prev30"
    ]
    for col in numeric_cols_to_fill:
        feature_vector[f"has_{col}"] = feature_vector[col].notna().astype(int)
        feature_vector[col] = feature_vector[col].fillna(0)

    # 6. Content Age Feature
    feature_vector["content_age_days_at_as_of"] = (
        pd.to_datetime(feature_vector["as_of_date"])
        - pd.to_datetime(feature_vector["content_created_date"])
    ).dt.days
    feature_vector["has_content_created_date"] = feature_vector["content_created_date"].notna().astype(int)
    feature_vector["content_age_days_at_as_of"] = feature_vector["content_age_days_at_as_of"].fillna(-1)
    feature_vector = feature_vector.drop(columns=["content_created_date"])

    # 7. Categorical Encoding
    categorical_cols = ["content_type", "competition_level", "main_intent"]
    for c in categorical_cols:
        print(f"Category counts for {c}:\n{feature_vector[c].value_counts(dropna=False)}")
    feature_vector_encoded = pd.get_dummies(feature_vector, columns=categorical_cols, dummy_na=False)

    # 8. Save to cache
    feature_vector_encoded.to_parquet(CACHE_PATH)
    print(f"Cached {len(feature_vector_encoded):,} finalized feature rows to {CACHE_PATH}")

print(f"Final feature_vector_encoded shape: {feature_vector_encoded.shape}")
print("First 5 rows of feature_vector_encoded:")
print(feature_vector_encoded.head())

Loaded 156,778 cached feature rows from work/outputs/features.parquet.
Final feature_vector_encoded shape: (156778, 50)
First 5 rows of feature_vector_encoded:
  as_of_date           client_hash_id           content_hash_id  \
0 2026-06-01  client_06d356715a8ff3b6  content_045a69625ade61a8   
1 2026-06-15  client_06d356715a8ff3b6  content_045a69625ade61a8   
2 2026-06-29  client_06d356715a8ff3b6  content_045a69625ade61a8   
3 2026-06-01  client_06d356715a8ff3b6  content_09063e5eace59644   
4 2026-06-15  client_06d356715a8ff3b6  content_09063e5eace59644   

   impressions_90d  impressions_last30  impressions_prev30  clicks_last30  \
0            115.0               115.0                 NaN            2.0   
1           2868.0              2868.0                 NaN            3.0   
2           4785.0              4785.0                 NaN            5.0   
3            605.0               605.0                 NaN           18.0   
4           9790.0              9790.0              

In [5]:
import pandas as pd
from pathlib import Path

CACHE_PATH = Path("work/outputs/features.parquet")

if CACHE_PATH.exists():
    loaded_features = pd.read_parquet(CACHE_PATH)
    print(f"Successfully loaded {len(loaded_features):,} rows from cached file: {CACHE_PATH}")
    print("First 5 rows of the loaded features:")
    print(loaded_features.head())
else:
    print(f"Error: Cache file not found at {CACHE_PATH}. Please ensure the previous cell ran successfully to generate and cache features.")

Successfully loaded 156,778 rows from cached file: work/outputs/features.parquet
First 5 rows of the loaded features:
  as_of_date           client_hash_id           content_hash_id  \
0 2026-06-01  client_06d356715a8ff3b6  content_045a69625ade61a8   
1 2026-06-15  client_06d356715a8ff3b6  content_045a69625ade61a8   
2 2026-06-29  client_06d356715a8ff3b6  content_045a69625ade61a8   
3 2026-06-01  client_06d356715a8ff3b6  content_09063e5eace59644   
4 2026-06-15  client_06d356715a8ff3b6  content_09063e5eace59644   

   impressions_90d  impressions_last30  impressions_prev30  clicks_last30  \
0            115.0               115.0                 NaN            2.0   
1           2868.0              2868.0                 NaN            3.0   
2           4785.0              4785.0                 NaN            5.0   
3            605.0               605.0                 NaN           18.0   
4           9790.0              9790.0                 NaN          254.0   

   clicks_prev30

In [6]:
print(CACHE_PATH)

work/outputs/features.parquet


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

In [7]:
feature_notes = pd.DataFrame([
    {"feature": "impressions_90d / impressions_last30 / impressions_prev30",
     "meaning": "GSC impressions summed over disjoint (as_of-90d,as_of], (as_of-30d,as_of], (as_of-60d,as_of-30d] windows",
     "missing_handling": "SUM with CASE->0: a true 0 means 'no impressions that window', not 'unknown'",
     "categorical": False, "available_at_predict_time": True},

    {"feature": "clicks_last30 / clicks_prev30",
     "meaning": "GSC clicks over the same last/prev 30d windows",
     "missing_handling": "SUM with CASE->0, same convention as impressions",
     "categorical": False, "available_at_predict_time": True},

    {"feature": "ctr_last30 / ctr_prev30 / impression_ratio_last_vs_prev30",
     "meaning": "clicks/impressions and impression-momentum ratios, derived from the rows above",
     "missing_handling": "NULLIF'd denominator -> NaN when 0 impressions, then has_-flag + fillna(0)",
     "categorical": False, "available_at_predict_time": True},

    {"feature": "avg_position_last30",
     "meaning": "mean GSC rank over the last 30d",
     "missing_handling": "0 treated as 'no data' in SQL NULLIF, then has_-flag + fillna(0)",
     "categorical": False, "available_at_predict_time": True},

    {"feature": "ga4_sessions_last30 / has_ga4_last90",
     "meaning": "GA4 sessions in the last 30d, gated on ga4_data_available IS TRUE",
     "missing_handling": "0 only counted when the flag confirms data genuinely exists; NULL-flag rows excluded from the sum, never zero-filled",
     "categorical": False, "available_at_predict_time": True},

    {"feature": "active_days_90d",
     "meaning": "distinct days with any fact row in the 90d window -- tells 'quiet content' apart from 'short/new history'",
     "missing_handling": "COUNT DISTINCT, 0 is a real value",
     "categorical": False, "available_at_predict_time": True},

    {"feature": "search_volume / competition / cpc / backlinks / category_count / keyword_char_count / keyword_token_count / url_char_count",
     "meaning": "keyword & content metadata from dim_content",
     "missing_handling": "has_<col> flag captures 'no keyword data for this content_type' BEFORE fillna(0) -- a blind fillna(0) alone would silently encode content_type",
     "categorical": False, "available_at_predict_time": True},

    {"feature": "content_type / competition_level / main_intent",
     "meaning": "categorical content/keyword metadata, one-hot encoded",
     "missing_handling": "explicit UNKNOWN_* sentinel category, never dropped or merged into an existing level",
     "categorical": True, "available_at_predict_time": True},

    {"feature": "content_age_days_at_as_of",
     "meaning": "as_of_date minus content_created_date",
     "missing_handling": "has_content_created_date flag; -1 sentinel when the date itself is missing",
     "categorical": False, "available_at_predict_time": True,
     "note": "legal by construction -- the WHERE guard already requires content_created_date <= as_of_date"},
])
display(feature_notes)

,feature,meaning,missing_handling,categorical,available_at_predict_time,note
0,impressions_90d / impressions_last30 / impress...,GSC impressions summed over disjoint (as_of-90...,SUM with CASE->0: a true 0 means 'no impressio...,False,True,NaN
1,clicks_last30 / clicks_prev30,GSC clicks over the same last/prev 30d windows,"SUM with CASE->0, same convention as impressions",False,True,NaN
2,ctr_last30 / ctr_prev30 / impression_ratio_las...,clicks/impressions and impression-momentum rat...,NULLIF'd denominator -> NaN when 0 impressions...,False,True,NaN
3,avg_position_last30,mean GSC rank over the last 30d,"0 treated as 'no data' in SQL NULLIF, then has...",False,True,NaN
4,ga4_sessions_last30 / has_ga4_last90,"GA4 sessions in the last 30d, gated on ga4_dat...",0 only counted when the flag confirms data gen...,False,True,NaN
5,active_days_90d,distinct days with any fact row in the 90d win...,"COUNT DISTINCT, 0 is a real value",False,True,NaN
6,search_volume / competition / cpc / backlinks ...,keyword & content metadata from dim_content,has_<col> flag captures 'no keyword data for t...,False,True,NaN
7,content_type / competition_level / main_intent,"categorical content/keyword metadata, one-hot ...","explicit UNKNOWN_* sentinel category, never dr...",True,True,NaN
8,content_age_days_at_as_of,as_of_date minus content_created_date,has_content_created_date flag; -1 sentinel whe...,False,True,legal by construction -- the WHERE guard alrea...


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [8]:
from sklearn.model_selection import GroupKFold, KFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

def cv_auc(X, y, groups=None, n_splits=5, random_state=0):
    """Out-of-fold AUC. Grouped (by `groups`) when given, else a random shuffled split."""
    if groups is not None:
        splitter = GroupKFold(n_splits=n_splits)
        split_args = (X, y, groups)
    else:
        splitter = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)
        split_args = (X,)
    scores = []
    for tr, te in splitter.split(*split_args):
        m = RandomForestClassifier(n_estimators=200, random_state=random_state, n_jobs=-1)
        m.fit(X.iloc[tr], y.iloc[tr])
        scores.append(roc_auc_score(y.iloc[te], m.predict_proba(X.iloc[te])[:, 1]))
    return sum(scores) / len(scores)

In [9]:
# Prepare the target variable (y) and grouping variable
y = feature_vector_encoded["is_declining"]
groups = feature_vector_encoded["content_hash_id"]

# Define non-feature columns that should not be in X
non_feature_cols = ["as_of_date", "client_hash_id", "content_hash_id", "is_declining"]

### Attack #1 — a genuine future-window column sitting in the table

`clicks_future30` is carried in `feature_vector` because it was needed to build the label — but it is exactly the kind of column the leakage taxonomy calls out first: information from strictly after the decision point. It must never reach `X`. Prove that by including it once, on purpose, and watching the score jump toward 1.0 — if the harness doesn't show the jump, the harness itself is broken and every later "clean" number is untrustworthy too.

In [11]:
# X will contain all features, including 'clicks_future30' for this test
X_with_leak = feature_vector_encoded.drop(columns=non_feature_cols)


# To speed up the calculation, we'll sample a subset of the data.
# This will allow a quicker demonstration of the leakage effect,
# though the absolute AUC values might differ slightly from a full run.
sample_fraction = 0.2  # Adjust this value (e.g., 0.1 for 10%, 0.5 for 50%) for speed/precision
print(f"Sampling {sample_fraction*100:.0f}% of the data for faster leakage test.")

# Ensure consistent sampling across X, y, and groups
sampled_indices = X_with_leak.sample(frac=sample_fraction, random_state=42).index

X_with_leak_sampled = X_with_leak.loc[sampled_indices]
y_sampled = y.loc[sampled_indices]
groups_sampled = groups.loc[sampled_indices]


# Calculate AUC with and without the leaked future column
# Use the sampled data for faster computation
auc_with_leak    = cv_auc(X_with_leak_sampled, y_sampled, groups_sampled)
auc_without_leak = cv_auc(X_with_leak_sampled.drop(columns=["clicks_future30"]), y_sampled, groups_sampled)

print(f"WITH clicks_future30 (illegal):    {auc_with_leak:.3f}")
print(f"WITHOUT clicks_future30 (correct): {auc_without_leak:.3f}")
print(f"Reading the gap. A sharp increase towards 1.0 indicates clear leakage.")

Sampling 20% of the data for faster leakage test.
WITH clicks_future30 (illegal):    0.999
WITHOUT clicks_future30 (correct): 0.922
Reading the gap. A sharp increase towards 1.0 indicates clear leakage.


### Split honesty — random vs grouped, and the base rate every score sits next to

The same `content_hash_id` appears at multiple `as_of_date`s. A random split lets the model see one date for a page in training and a neighboring date for the same page in the test fold — it can partly memorize the page instead of learning the general pattern. `GroupKFold` on `content_hash_id` closes that door. Report both, and never a metric without its base rate.

In [13]:
# The clean feature set (excluding the future-window column)
X_clean = X_with_leak.drop(columns=["clicks_future30"])


# To speed up the calculation, we'll sample a subset of the data.
# This will allow a quicker demonstration of split honesty,
# though the absolute AUC values might differ slightly from a full run.
sample_fraction = 0.2  # Adjust this value (e.g., 0.1 for 10%, 0.5 for 50%) for speed/precision
print(f"Sampling {sample_fraction*100:.0f}% of the data for faster split honesty test.")

# Ensure consistent sampling across X, y, and groups
sampled_indices_clean = X_clean.sample(frac=sample_fraction, random_state=42).index

X_clean_sampled = X_clean.loc[sampled_indices_clean]
y_sampled_clean = y.loc[sampled_indices_clean]
groups_sampled_clean = groups.loc[sampled_indices_clean]


base_rate = y_sampled_clean.mean() # Calculate base rate from sampled data as well for consistency

# Use the sampled data for faster computation
auc_random  = cv_auc(X_clean_sampled, y_sampled_clean, groups=None)
auc_grouped = cv_auc(X_clean_sampled, y_sampled_clean, groups=groups_sampled_clean)

print(f"Base rate (positive class, is_declining=1): {base_rate:.3f}")
print(f"Random-split AUC:  {auc_random:.3f}")
print(f"Grouped-split AUC: {auc_grouped:.3f}")
print(f"Gap (random - grouped): {auc_random - auc_grouped:.3f}  <- how much the random split let the model memorize a page")

Sampling 20% of the data for faster split honesty test.
Base rate (positive class, is_declining=1): 0.719
Random-split AUC:  0.922
Grouped-split AUC: 0.922
Gap (random - grouped): -0.000  <- how much the random split let the model memorize a page


### Sanity check: is a decision-derived flag masquerading as a predictor?

`provider_used` / `model_used` record a choice an existing content-generation system already made. If the decline rate varies sharply by provider, that is a fact about the current system's past decisions, not a causal driver a new model should condition on — it's a baseline to beat, never an input (taxonomy #3). It is intentionally NOT in `feature_vector`; the check below joins it back in only to audit it, never to add it to X.

In [14]:
# Re-join dim_content to get 'provider_used' for auditing
provider_decline_rates = con.sql(f"""
    SELECT
        d.provider_used,
        AVG(fv.is_declining) AS decline_rate,
        COUNT(*) AS n
    FROM feature_vector fv
    JOIN {TABLES['dim_content']} d USING (client_hash_id, content_hash_id)
    GROUP BY 1
    ORDER BY n DESC
""").df()
display(provider_decline_rates)

# Audit for 'model_used' as well
model_decline_rates = con.sql(f"""
    SELECT
        d.model_used,
        AVG(fv.is_declining) AS decline_rate,
        COUNT(*) AS n
    FROM feature_vector fv
    JOIN {TABLES['dim_content']} d USING (client_hash_id, content_hash_id)
    GROUP BY 1
    ORDER BY n DESC
""").df()
display(model_decline_rates)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,provider_used,decline_rate,n
0,None,0.726242,78080
1,google,0.724888,41016
2,gemini-generate-content,0.705884,36030
3,openai,0.797848,1301
4,vexxskincare---openai,0.841004,239
5,Flyrank Dev - OpenAI,0.901408,71
6,poshe---openai,0.923077,39
7,BalancedFitnessGear---open-ai,1.000000,2


,model_used,decline_rate,n
0,gemini-3-flash-preview,0.707858,129519
1,None,0.801287,15545
2,gemini-2.5-flash,0.733052,5679
3,gpt-4o-mini,0.817079,4450
4,gpt-5-mini,0.807606,1341
5,unknown,0.786885,244


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

In [15]:
EXCLUDED_FIELDS = {
    "last_optimized_date": "Current-state field; historical availability is not reconstructed",
    "content_updated_date": "Current-state field; historical availability is not reconstructed",
    "optimization_eligible_date": "Historical availability is not reconstructed",
    "is_published": "Current-state snapshot",
    "is_deleted": "Current-state snapshot",
    "provider_used": "Post-decision/current optimization information",
    "model_used": "Post-decision/current optimization information",
}

for field, reason in EXCLUDED_FIELDS.items():
    print(f"{field}: {reason}")

last_optimized_date: Current-state field; historical availability is not reconstructed
content_updated_date: Current-state field; historical availability is not reconstructed
optimization_eligible_date: Historical availability is not reconstructed
is_published: Current-state snapshot
is_deleted: Current-state snapshot
provider_used: Post-decision/current optimization information
model_used: Post-decision/current optimization information


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.